# Track A 00: Freeze and Verify Historical Llama Evidence

This notebook implements gate **A0**. It treats the completed Llama v5 directory as immutable historical evidence and writes checksums to a separate Drive directory.

**Safety boundary:** this notebook never writes inside `CSE407_llama_v5_final/`.

## Run Order

1. Use a normal Colab CPU runtime; a GPU is not required.
2. Confirm `CSE407_llama_v5_final/` is in the root of My Drive.
3. Run all cells once to create the freeze manifest.
4. Run again at any time to verify 114 scientific artifact files and 95 power traces. A repository copy may report 115 when it also contains the non-evidence `.gitignore`.

If verification fails, stop. Do not regenerate the freeze over changed evidence.

## Integrity Functions

The checks cover primary row counts, experiment fingerprint consistency, all 380 structural passes, exact trace names, byte sizes, and SHA-256 hashes.

In [ ]:
import csv
import hashlib
import json
from pathlib import Path

PRIMARY_ROWS = {
    "v5_final_energy_results.csv": 75,
    "v5_final_quality_results.csv": 225,
    "v5_final_sensitivity_energy_4096.csv": 20,
    "v5_final_sensitivity_quality_4096.csv": 60,
    "v5_final_turboquant_distortion.csv": 6,
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_csv(path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def expected_trace_names(rows):
    names = set()
    for row in rows:
        variant = "".join(
            character if character.isalnum() or character in "_.-" else "_"
            for character in row["variant"]
        )
        names.add(
            f"{variant}_ctx{int(float(row['context']))}_block{int(float(row['block']))}.json"
        )
    return names

def validate_evidence():
    assert ARTIFACT_ROOT.is_dir(), f"Missing artifact directory: {ARTIFACT_ROOT}"
    environment = json.loads((ARTIFACT_ROOT / "v5_final_environment.json").read_text())
    validation = json.loads((ARTIFACT_ROOT / "v5_final_validation.json").read_text())
    assert validation["passed"]
    assert environment["schema_version"] == validation["schema_version"]
    tables = {}
    for filename, expected in PRIMARY_ROWS.items():
        rows = read_csv(ARTIFACT_ROOT / filename)
        assert len(rows) == expected, (filename, len(rows), expected)
        assert {row["fingerprint"] for row in rows} == {validation["fingerprint"]}
        tables[filename] = rows
    execution_rows = sum(
        (rows for name, rows in tables.items() if name != "v5_final_turboquant_distortion.csv"),
        [],
    )
    assert len(execution_rows) == 380
    assert all(row["structural_pass"].casefold() == "true" for row in execution_rows)
    energy_rows = tables["v5_final_energy_results.csv"] + tables["v5_final_sensitivity_energy_4096.csv"]
    expected_traces = expected_trace_names(energy_rows)
    observed_traces = {path.name for path in (ARTIFACT_ROOT / "power_traces").glob("*.json")}
    assert len(observed_traces) == 95 and observed_traces == expected_traces
    return {
        "schema_version": validation["schema_version"],
        "fingerprint": validation["fingerprint"],
        "session_id": environment["session_id"],
        "model_fp16_revision": environment["model_fp16_revision"],
        "model_awq_revision": environment["model_awq_revision"],
        "turboquant_source_sha256": environment["turboquant_source_sha256"],
        "primary_execution_rows": 380,
        "power_trace_count": 95,
    }

def collect_files():
    return [
        {
            "path": path.relative_to(ARTIFACT_ROOT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(item for item in ARTIFACT_ROOT.rglob("*") if item.is_file())
    ]

def current_manifest():
    facts = validate_evidence()
    files = collect_files()
    return {
        "manifest_schema": "cse407-artifact-manifest-v1",
        "experiment_id": "llama-3.2-3b-v5-reference",
        "artifact_root": str(ARTIFACT_ROOT),
        "status": "IMMUTABLE_HISTORICAL_EVIDENCE",
        "policy": "Never edit or overwrite files under artifact_root.",
        "facts": facts,
        "inventory": {
            "file_count": len(files),
            "total_bytes": sum(record["bytes"] for record in files),
            "power_trace_count": facts["power_trace_count"],
        },
        "files": files,
    }

## Mount Drive and Freeze or Verify

The first run creates `/MyDrive/CSE407_track_a_registry/llama_v5_artifact_manifest.json`. Later runs compare against it and do not overwrite it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/CSE407_llama_v5_final")
FREEZE_DIR = Path("/content/drive/MyDrive/CSE407_track_a_registry")
FREEZE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = FREEZE_DIR / "llama_v5_artifact_manifest.json"

observed = current_manifest()
if MANIFEST_PATH.exists():
    frozen = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    assert frozen["facts"] == observed["facts"], "Historical metadata changed"
    assert frozen["files"] == observed["files"], "Historical artifact bytes changed"
    action = "VERIFIED_EXISTING_FREEZE"
else:
    temporary = MANIFEST_PATH.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(observed, indent=2) + "\n", encoding="utf-8")
    temporary.replace(MANIFEST_PATH)
    action = "CREATED_INITIAL_FREEZE"

print(action)
print(json.dumps(observed["facts"], indent=2))
print("Files:", observed["inventory"]["file_count"])
print("Manifest:", MANIFEST_PATH)

## Acceptance Gate

Proceed only when the output reports either `CREATED_INITIAL_FREEZE` or `VERIFIED_EXISTING_FREEZE`, fingerprint `8d27af8c9ee94de3`, 380 execution rows, 95 traces, and 114 scientific artifact files. A repository copy may report 115 only because it includes `.gitignore`.